# Deep learning and representation: memory arithmetic and the input pipeline

Exploration only. The service depends on `nw/semantic`; this notebook imports it.

## Memory arithmetic, from the real parameter count

In [1]:
from nw.semantic.model import ModelSpec, build_encoder, apply_lora, count_parameters
model, tok = build_encoder(ModelSpec())
total, _ = count_parameters(model)
print(f"parameters: {total/1e6:.1f}M")
for name, bytes_per in [("weights fp32", 4), ("+ gradients", 8), ("+ Adam moments", 16)]:
    print(f"{name:16s} {total * bytes_per / 1e9:.2f} GB")
model = apply_lora(model, ModelSpec())
total, trainable = count_parameters(model)
print(f"LoRA trainable: {trainable/1e6:.2f}M ({100*trainable/total:.2f}%), optimizer state {trainable*8/1e6:.0f} MB")

/Users/shadabhussain/Downloads/Courses/AI Engineering/solution/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8544.11it/s]


[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


parameters: 66.4M
weights fp32     0.27 GB
+ gradients      0.53 GB
+ Adam moments   1.06 GB
LoRA trainable: 0.19M (0.29%), optimizer state 2 MB


## Where the time goes: padding

In [2]:
import time, torch
from nw.semantic.data import load_rows, make_loader
rows = load_rows("../data/tickets.jsonl", "train")[:512]
for fixed in (False, True):
    from nw.semantic.data import Collate
    loader = make_loader(rows, tok, batch_size=32)
    if fixed:
        loader.collate_fn = lambda b, _c=Collate(tok): {**_c(b), "input_ids": torch.nn.functional.pad(_c(b)["input_ids"], (0, 256 - _c(b)["input_ids"].shape[1]))}
    t0 = time.perf_counter(); tokens = 0
    for batch in loader:
        tokens += batch["input_ids"].numel()
    print(f"{'fixed 256' if fixed else 'dynamic'} padding: {tokens:,} tokens through the encoder, {time.perf_counter()-t0:.2f}s to tokenise")

dynamic padding: 103,904 tokens through the encoder, 0.07s to tokenise


fixed 256 padding: 131,072 tokens through the encoder, 0.22s to tokenise


## Per-head metrics on the test split, from the exported graph

In [3]:
import json, numpy as np
from nw.semantic.export import OnnxEncoder, load_finetuned
from nw.semantic.data import TAGS, multi_hot
from nw.triage.features import PRIORITIES, ticket_text
from sklearn.metrics import confusion_matrix, f1_score
rows = load_rows("../data/tickets.jsonl", "test")
model, tok, meta = load_finetuned("../artifacts/semantic")
onnx = OnnxEncoder("../artifacts/semantic/latest/model.int8.onnx", tok, meta["max_length"])
thr = np.load("../artifacts/semantic/latest/tag_thresholds.npy")
tag_p, prio_l = [], []
for i in range(0, len(rows), 64):
    a, b, _ = onnx.run([ticket_text(r["subject"], r["body"]) for r in rows[i:i+64]])
    tag_p.append(1/(1+np.exp(-a))); prio_l.append(b)
tag_p, prio_l = np.concatenate(tag_p), np.concatenate(prio_l)
y_prio = np.array([PRIORITIES.index(r["priority"]) for r in rows]); y_tags = np.stack([multi_hot(r["tags"]).numpy() for r in rows])
print("priority confusion (rows true):"); print(confusion_matrix(y_prio, prio_l.argmax(1)))
per_tag = f1_score(y_tags, tag_p >= thr, average=None, zero_division=0)
for t, f in sorted(zip(TAGS, per_tag), key=lambda x: x[1])[:8]:
    print(f"weakest tag {t:14s} F1 {f:.2f}  n={int(y_tags[:, TAGS.index(t)].sum())}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 28079.96it/s]


[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


priority confusion (rows true):
[[ 22   9   0   0]
 [  7 193  47  10]
 [  0  69 176  84]
 [  0   3  37 137]]
weakest tag Security       F1 0.00  n=9
weakest tag Breach         F1 0.00  n=8
weakest tag Compliance     F1 0.00  n=0
weakest tag GDPR           F1 0.00  n=0
weakest tag Data Loss      F1 0.27  n=9
weakest tag Outage         F1 0.31  n=5
weakest tag Feedback       F1 0.33  n=155
weakest tag Documentation  F1 0.34  n=162


## Questions to bring back

- Which tags are weakest, and are they rare or ambiguous?
- Where do German tickets land in the priority confusion?